In [ ]:
from keras.layers import Dense,Flatten
from keras.applications.mobilenet_v2 import MobileNetV2
import numpy as np
import pandas as pd

In [ ]:
train_path = '/content/drive/MyDrive/Brain_Tumor/Training'
test_path = '/content/drive/MyDrive/Brain_Tumor/Testing'

In [ ]:
img_size = [224,224]

In [ ]:
v2 = MobileNetV2(input_shape=img_size+[3],weights='imagenet',include_top=False)

In [ ]:
for layer in v2.layers:
  layer.trainable = False

In [ ]:
X = Flatten()(v2.output)
out = Dense(4,activation='softmax')(X)

In [ ]:
from keras.models import Model

In [ ]:
model = Model(inputs=v2.inputs,outputs=out)

In [ ]:
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Conv1 (Conv2D)      │ (None, 112, 112,  │        864 │ input_layer[0][0] │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bn_Conv1            │ (None, 112, 112,  │        128 │ Conv1[0][0]       │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Conv1_relu (ReLU)   │ (None, 112, 112,  │          0 │ bn_Conv1[0][0]    │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 112, 112,  │        288 │ Conv1_relu[0][0]  │
│ (DepthwiseConv2D)   │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 112, 112,  │        128 │ expanded_conv_de… │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 112, 112,  │          0 │ expanded_conv_de… │
│ (ReLU)              │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_proj… │ (None, 112, 112,  │        512 │ expanded_conv_de… │
│ (Conv2D)            │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_proj… │ (None, 112, 112,  │         64 │ expanded_conv_pr… │
│ (BatchNormalizatio… │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand      │ (None, 112, 112,  │      1,536 │ expanded_conv_pr… │
│ (Conv2D)            │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand_BN   │ (None, 112, 112,  │        384 │ block_1_expand[0… │
│ (BatchNormalizatio… │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand_relu │ (None, 112, 112,  │          0 │ block_1_expand_B… │
│ (ReLU)              │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_pad         │ (None, 113, 113,  │          0 │ block_1_expand_r… │
│ (ZeroPadding2D)     │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise   │ (None, 56, 56,    │        864 │ block_1_pad[0][0] │
│ (DepthwiseConv2D)   │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise_… │ (None, 56, 56,    │        384 │ block_1_depthwis… │
│ (BatchNormalizatio… │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise_… │ (None, 56, 56,    │          0 │ block_1_depthwis… │
│ (ReLU)              │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_project     │ (None, 56, 56,    │      2,304 │ block_1_depthwis

 Total params: 2,508,868 (9.57 MB)

 Trainable params: 250,884 (980.02 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

In [ ]:
model.compile(optimizer='adam',loss='categorical_crossentropy',metrics=['accuracy'])

In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

In [ ]:
train_gen = ImageDataGenerator(rescale=1/255,zoom_range=0.2)
test_gen = ImageDataGenerator(rescale=1/255)

In [ ]:
train_set = train_gen.flow_from_directory(train_path,
                                          target_size=(224,224),
                                          class_mode= 'categorical',
                                          batch_size=32,shuffle=False)

Found 5529 images belonging to 4 classes.


In [ ]:
test_set = train_gen.flow_from_directory(test_path,
                                          target_size=(224,224),
                                          class_mode= 'categorical',
                                          shuffle=False)

Found 1311 images belonging to 4 classes.


In [ ]:
model.fit(train_set,validation_data=test_set,epochs=3)

Epoch 1/3
173/173 ━━━━━━━━━━━━━━━━━━━━ 3738s 22s/step - accuracy: 0.7061 - loss: 9.7693 - val_accuracy: 0.5622 - val_loss: 20.9032
Epoch 2/3
173/173 ━━━━━━━━━━━━━━━━━━━━ 106s 613ms/step - accuracy: 0.8112 - loss: 4.7903 - val_accuracy: 0.8665 - val_loss: 2.9131
Epoch 3/3
173/173 ━━━━━━━━━━━━━━━━━━━━ 108s 622ms/step - accuracy: 0.9014 - loss: 1.7892 - val_accuracy: 0.8558 - val_loss: 1.9320


In [ ]:
y_pred = model.predict(test_set)
y_pred

41/41 ━━━━━━━━━━━━━━━━━━━━ 30s 622ms/step


array([[1.0000000e+00, 7.7660524e-24, 3.1574579e-33, 1.0010876e-41],
       [1.0000000e+00, 6.7598646e-23, 3.9013882e-29, 2.0792438e-26],
       [1.0000000e+00, 8.5655238e-26, 2.5734356e-34, 1.7146849e-39],
       ...,
       [2.3793350e-23, 8.6456231e-11, 4.2618345e-22, 1.0000000e+00],
       [2.4540199e-16, 5.9324411e-26, 1.0615311e-36, 1.0000000e+00],
       [5.1520597e-30, 3.4861672e-21, 7.1231400e-37, 1.0000000e+00]],
      dtype=float32)

In [ ]:
from sklearn.metrics import classification_report
import os

In [ ]:
pred = y_pred.argmax(axis=1)

In [ ]:
print(classification_report(test_set.classes,pred))

              precision    recall  f1-score   support

           0       0.83      0.90      0.86       300
           1       0.74      0.85      0.79       306
           2       0.99      0.87      0.93       405
           3       0.94      0.87      0.90       300

    accuracy                           0.87      1311
   macro avg       0.87      0.87      0.87      1311
weighted avg       0.88      0.87      0.88      1311



In [ ]:
brain_tum = os.listdir('/content/drive/MyDrive/Brain_Tumor/Testing')
brain_tum.sort()
brain_tum

['glioma', 'meningioma', 'notumor', 'pituitary']

In [ ]:
from skimage.io import imread
from skimage.transform import resize

In [ ]:
def image(img):
  read = imread(img)

  img_resize = resize(read,(224,224,3))
  img_shape = img_resize.reshape(1,224,224,3)

  pred= model.predict(img_shape)

  ind = pred.argmax()
  pred = brain_tum[ind]

  print(pred)

In [ ]:
image('/content/drive/MyDrive/Brain_Tumor/Testing/meningioma/Te-meTr_0002.jpg')

1/1 ━━━━━━━━━━━━━━━━━━━━ 12s 12s/step
meningioma
